# Cloudflare Tunnel Health Monitor

Monitoring cho Cloudflare Tunnel kết nối đến A100 Ollama service.

**Tunnel URL:** https://grew-hypothesis-mothers-flooring.trycloudflare.com

**Purpose:** Expose Ollama service từ A100 Docker JupyterLab ra internet

In [ ]:
# Cell 1: Install dependencies
!pip install requests pandas matplotlib

In [ ]:
# Cell 2: Tunnel health check functions
import requests
import time
from datetime import datetime

TUNNEL_URL = 'https://grew-hypothesis-mothers-flooring.trycloudflare.com'

def check_tunnel_health():
    """Check if Cloudflare Tunnel is accessible"""
    try:
        start_time = time.time()
        response = requests.get(f'{TUNNEL_URL}/api/tags', timeout=10)
        elapsed = (time.time() - start_time) * 1000
        
        if response.ok:
            return {
                'status': 'healthy',
                'response_time_ms': elapsed,
                'status_code': response.status_code,
                'timestamp': datetime.now().isoformat()
            }
        else:
            return {
                'status': 'unhealthy',
                'response_time_ms': elapsed,
                'status_code': response.status_code,
                'error': f'HTTP {response.status_code}',
                'timestamp': datetime.now().isoformat()
            }
    except requests.exceptions.Timeout:
        return {
            'status': 'timeout',
            'error': 'Request timeout (>10s)',
            'timestamp': datetime.now().isoformat()
        }
    except requests.exceptions.ConnectionError:
        return {
            'status': 'down',
            'error': 'Connection failed - tunnel may be down',
            'timestamp': datetime.now().isoformat()
        }
    except Exception as e:
        return {
            'status': 'error',
            'error': str(e),
            'timestamp': datetime.now().isoformat()
        }

def check_tunnel_latency(samples=5):
    """Measure tunnel latency over multiple samples"""
    latencies = []
    
    for _ in range(samples):
        try:
            start_time = time.time()
            response = requests.get(f'{TUNNEL_URL}/api/tags', timeout=10)
            elapsed = (time.time() - start_time) * 1000
            
            if response.ok:
                latencies.append(elapsed)
        except:
            pass
        
        time.sleep(1)
    
    if latencies:
        return {
            'avg_ms': sum(latencies) / len(latencies),
            'min_ms': min(latencies),
            'max_ms': max(latencies),
            'samples': len(latencies),
            'timestamp': datetime.now().isoformat()
        }
    else:
        return {
            'error': 'All requests failed',
            'timestamp': datetime.now().isoformat()
        }

print("✅ Tunnel health check functions loaded")

In [ ]:
# Cell 3: Continuous tunnel monitoring
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output

tunnel_history = []
max_history = 100

print("🚀 Starting Cloudflare Tunnel monitoring... (Press Interrupt to stop)\n")

try:
    while True:
        health = check_tunnel_health()
        tunnel_history.append(health)
        
        if len(tunnel_history) > max_history:
            tunnel_history.pop(0)
        
        clear_output(wait=True)
        
        # Plot latency graph
        df = pd.DataFrame([h for h in tunnel_history if h['status'] == 'healthy'])
        
        if not df.empty:
            plt.figure(figsize=(12, 4))
            plt.plot(df.index, df['response_time_ms'], color='#3498db', linewidth=2)
            plt.fill_between(df.index, df['response_time_ms'], alpha=0.3, color='#3498db')
            plt.title('Cloudflare Tunnel Response Time', fontsize=14, fontweight='bold')
            plt.ylabel('Response Time (ms)')
            plt.xlabel('Sample')
            plt.grid(True, alpha=0.3)
            plt.axhline(y=1000, color='orange', linestyle='--', alpha=0.5, label='Warning: 1000ms')
            plt.axhline(y=2000, color='red', linestyle='--', alpha=0.5, label='Critical: 2000ms')
            plt.legend()
            plt.tight_layout()
            plt.show()
        
        # Display current status
        print("🌐 Cloudflare Tunnel Status")
        print("=" * 60)
        print(f"URL: {TUNNEL_URL}")
        print(f"Timestamp: {health['timestamp']}")
        print(f"Status: {health['status'].upper()}")
        
        if health['status'] == 'healthy':
            print(f"✅ Tunnel is accessible")
            print(f"Response Time: {health['response_time_ms']:.2f}ms")
            print(f"HTTP Status: {health['status_code']}")
            
            # Latency warning
            if health['response_time_ms'] > 2000:
                print("\n⚠️ CRITICAL: Very high latency (>2000ms)")
            elif health['response_time_ms'] > 1000:
                print("\n⚠️ WARNING: High latency (>1000ms)")
        else:
            print(f"❌ Tunnel is {health['status']}")
            print(f"Error: {health.get('error', 'Unknown')}")
        
        print("=" * 60)
        
        # Calculate uptime and avg latency
        if len(tunnel_history) > 1:
            healthy_count = sum(1 for h in tunnel_history if h['status'] == 'healthy')
            uptime_percent = (healthy_count / len(tunnel_history)) * 100
            
            if not df.empty:
                avg_latency = df['response_time_ms'].mean()
                print(f"\n📊 Stats (last {len(tunnel_history)} checks):")
                print(f"  Uptime: {uptime_percent:.1f}%")
                print(f"  Avg Latency: {avg_latency:.2f}ms")
        
        time.sleep(60)  # Check every 60 seconds
        
except KeyboardInterrupt:
    print("\n\n🛑 Monitoring stopped")
    
    if tunnel_history:
        df = pd.DataFrame(tunnel_history)
        filename = f"tunnel_health_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, index=False)
        print(f"📁 Health history saved to: {filename}")

In [ ]:
# Cell 4: Quick tunnel check
print("🌐 Cloudflare Tunnel Quick Check\n")

health = check_tunnel_health()

print(f"URL: {TUNNEL_URL}")
print(f"Status: {health['status'].upper()}")

if health['status'] == 'healthy':
    print(f"✅ Tunnel is accessible")
    print(f"Response Time: {health['response_time_ms']:.2f}ms")
else:
    print(f"❌ Tunnel is {health['status']}")
    print(f"Error: {health.get('error')}")

In [ ]:
# Cell 5: Latency benchmark
print("⚡ Cloudflare Tunnel Latency Benchmark\n")
print("Running 10 samples...\n")

result = check_tunnel_latency(samples=10)

if 'error' not in result:
    print(f"Latency Stats:")
    print(f"  Average: {result['avg_ms']:.2f}ms")
    print(f"  Min: {result['min_ms']:.2f}ms")
    print(f"  Max: {result['max_ms']:.2f}ms")
    print(f"  Samples: {result['samples']}/10")
    
    if result['avg_ms'] < 500:
        print("\n✅ Excellent latency")
    elif result['avg_ms'] < 1000:
        print("\n✅ Good latency")
    elif result['avg_ms'] < 2000:
        print("\n⚠️ Acceptable latency")
    else:
        print("\n❌ Poor latency - investigate network issues")
else:
    print(f"❌ Benchmark failed: {result['error']}")